# Final Project - BeautifulSoup Documentation Analytics System

**Thành viên 1: HTML Collector & Section Extractor**

**Project goal:** collect and parse the BeautifulSoup documentation page, then extract section metadata.

**Target URL:** https://www.crummy.com/software/BeautifulSoup/bs4/doc/

**Two ways to run this project:**
- **Notebook** (this file) - step-by-step interactive analysis
- **Script** - run `python src/main.py` for a headless pipeline

Features implemented:

1. Web Page Collector  
2. HTML Parser  
3. Section Extractor

## Project Architecture

The core logic lives in the `src/` Python package:

| File | Description |
|------|-------------|
| `src/collector.py` | Feature 1: Downloads and saves raw HTML |
| `src/parser.py` | Feature 2: Parses HTML with BeautifulSoup |
| `src/extractor.py` | Feature 3: Extracts documentation sections |

The cells below mirror the same logic step-by-step for transparency.

## 1. Import Required Libraries

In [1]:
import os
import re
import requests
import pandas as pd
from bs4 import BeautifulSoup

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)


## 2. Create Project Folders

In [2]:
# Anchor the working directory to the project root so all relative paths
# (data/) resolve correctly regardless of where Jupyter was started.
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

print("Folders created successfully.")


Project root: C:\FinalPDS\FinalProjectPDS301m
Folders created successfully.


## 3. Feature 1 – Web Page Collector

This step downloads the BeautifulSoup documentation page using the `requests` library and saves the raw HTML file into `data/raw/beautifulsoup_doc.html`.

In [3]:
url = "https://www.crummy.com/software/BeautifulSoup/bs4/doc/"
raw_html_path = "data/raw/beautifulsoup_doc.html"

response = requests.get(url, timeout=20)
print("Status code:", response.status_code)

if response.status_code == 200:
    with open(raw_html_path, "w", encoding="utf-8") as file:
        file.write(response.text)
    print("Raw HTML saved to:", raw_html_path)
else:
    raise Exception("Failed to download the documentation page.")


Status code: 200
Raw HTML saved to: data/raw/beautifulsoup_doc.html


## 4. Feature 2 – HTML Parser

This step reads the saved HTML file and parses it using BeautifulSoup.

In [4]:
with open(raw_html_path, "r", encoding="utf-8") as file:
    html = file.read()

soup = BeautifulSoup(html, "html.parser")

page_title = soup.title.get_text(strip=True) if soup.title else "No title found"
print("Page title:", page_title)


Page title: Beautiful Soup Documentation — Beautiful Soup 4.14.3 documentation


## 5. Helper Functions

These functions help identify sections, clean text, classify links, and find the section title for each extracted item.

In [5]:
heading_tags = ["h1", "h2", "h3"]


def clean_text(text):
    """Clean extra spaces and line breaks from text."""
    if text is None:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def get_content_until_next_heading(heading):
    """Return all sibling elements after a heading until the next h1, h2, or h3."""
    content = []
    for sibling in heading.find_next_siblings():
        if sibling.name in heading_tags:
            break
        content.append(sibling)
    return content


## 6. Feature 3 – Section Extractor

Required output file: `data/processed/sections.csv`

Required columns:

- `section_id`
- `section_level`
- `section_title`
- `section_text`
- `word_count`
- `code_block_count`
- `link_count`

In [6]:
headings = soup.find_all(heading_tags)
print("Number of headings found:", len(headings))

sections_data = []

for index, heading in enumerate(headings, start=1):
    section_title = clean_text(heading.get_text(" ", strip=True))
    section_level = heading.name
    section_elements = get_content_until_next_heading(heading)

    section_text_parts = []
    code_block_count = 0
    link_count = 0

    for element in section_elements:
        section_text_parts.append(clean_text(element.get_text(" ", strip=True)))
        code_block_count += len(element.find_all(["pre", "code"]))
        link_count += len(element.find_all("a"))

    section_text = clean_text(" ".join(section_text_parts))
    word_count = len(section_text.split())

    sections_data.append({
        "section_id": index,
        "section_level": section_level,
        "section_title": section_title,
        "section_text": section_text,
        "word_count": word_count,
        "code_block_count": code_block_count,
        "link_count": link_count
    })

sections_df = pd.DataFrame(sections_data)
sections_path = "data/processed/sections.csv"
sections_df.to_csv(sections_path, index=False, encoding="utf-8-sig")

print("Sections saved to:", sections_path)
sections_df.head()


Number of headings found: 113
Sections saved to: data/processed/sections.csv


,section_id,section_level,section_title,section_text,word_count,code_block_count,link_count
0,1,h1,Beautiful Soup Documentation ¶,Beautiful Soup is a Python library for pulling data out of HTML and XML files. It works with your favorite parser to...,310,0,14
1,2,h2,Getting help ¶,"If you have questions about Beautiful Soup, or run into problems, send mail to the discussion group . If your proble...",97,0,4
2,3,h3,API documentation ¶,"This document is written like an instruction manual, but you can also read traditional API documentation generated f...",43,0,1
3,4,h1,Quick Start ¶,Here's an HTML document I'll be using as an example throughout this document. It's part of a story from Alice in Won...,422,6,0
4,5,h1,Installing Beautiful Soup ¶,"If you're using a recent version of Debian or Ubuntu Linux, you can install Beautiful Soup with the system package m...",431,18,6


## Extracted Data Summary

In [7]:
summary_data = {
    "Dataset": ["sections.csv"],
    "Rows": [len(sections_df)],
    "Columns": [sections_df.shape[1]]
}

summary_df = pd.DataFrame(summary_data)
print(f"Total sections extracted: {len(sections_df)}")
print(f"Columns: {list(sections_df.columns)}")
summary_df


Total sections extracted: 113
Columns: ['section_id', 'section_level', 'section_title', 'section_text', 'word_count', 'code_block_count', 'link_count']


,Dataset,Rows,Columns
0,sections.csv,113,7


## Final Report Content

### Dataset Overview

This project analyzes the official BeautifulSoup documentation page. The original dataset is a raw HTML page downloaded directly from the target website. After parsing the page, the system extracts structured section data.

### Scraping Method

The system uses `requests` to send an HTTP request to the target URL. If the status code is 200, the raw HTML is saved to `data/raw/beautifulsoup_doc.html`. Then, BeautifulSoup parses the HTML content and extracts section information from headings.

### Extracted Data Summary

The project generates one processed CSV file:

- `sections.csv`: stores documentation section details including section title, level, text, word count, code block count, and link count.

### Key Findings

- The documentation is divided into structured sections (h1, h2, h3).
- Each section contains a varying number of words, code blocks, and links.
- Word count, code block count, and link count are computed per section for further analysis.

### Limitations

- The project analyzes only one documentation page.
- Section content is extracted based on heading tag boundaries.
- If the documentation website changes in the future, the results may also change.

### Conclusion

This project successfully builds a Python-based documentation analytics system. It collects raw HTML data, parses the documentation, and extracts structured section datasets for downstream analysis.


## Submission Checklist (Thành viên 1)

Before submitting, make sure these files are available:

- `notebooks/FinalProject_BeautifulSoup_Analysis.ipynb`
- `data/raw/beautifulsoup_doc.html`
- `data/processed/sections.csv`
- `src/collector.py`
- `src/parser.py`
- `src/extractor.py`
- `src/main.py`
- `README.md`
- `requirements.txt`

To export the notebook to PDF, use Jupyter Notebook menu:

`File` → `Save and Export Notebook As` → `PDF`